In [1]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
import random
import re

# ==============================
# Device Setup
# ==============================
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==============================
# Haar Cascade Face Detection
# ==============================
haar_cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
face_cascade = cv2.CascadeClassifier(haar_cascade_path)

def detect_face_haar(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(40, 40))
    if len(faces) > 0:
        x, y, w, h = faces[0]
        face = image[y:y+h, x:x+w]
        face = cv2.resize(face, (112, 112))
        return face
    else:
        return np.zeros((112, 112, 3), dtype=np.uint8)

# ==============================
# Dataset
# ==============================
class SiameseFaceDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.pairs = []
        self.transform = transform

        for root, _, _ in os.walk(root_dir):
            match = re.search(r'label_(\d+)', root)
            if not match:
                continue
            label = int(match.group(1))
            im0 = next((f for f in os.listdir(root) if f.startswith("im_0_")), None)
            im1 = next((f for f in os.listdir(root) if f.startswith("im_1_")), None)
            if im0 and im1:
                img0_path = os.path.join(root, im0)
                img1_path = os.path.join(root, im1)
                self.pairs.append((img0_path, img1_path, label))

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        img0_path, img1_path, label = self.pairs[idx]
        img0 = cv2.imread(img0_path)
        img1 = cv2.imread(img1_path)
        img0 = cv2.cvtColor(img0, cv2.COLOR_BGR2RGB)
        img1 = cv2.cvtColor(img1, cv2.COLOR_BGR2RGB)
        img0 = detect_face_haar(img0)
        img1 = detect_face_haar(img1)
        if self.transform:
            img0 = self.transform(image=img0)['image']
            img1 = self.transform(image=img1)['image']
        return img0.to(torch.float32), img1.to(torch.float32), torch.tensor(label, dtype=torch.float32)

# ==============================
# Transforms
# ==============================
transform = A.Compose([
    A.Resize(112, 112),
    A.Normalize(),
    ToTensorV2()
])

# ==============================
# Model
# ==============================
class SiameseNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        with torch.no_grad():
            dummy = self.backbone(torch.zeros(1, 3, 112, 112))
            self.flattened = dummy.view(1, -1).shape[1]
        self.fc = nn.Sequential(
            nn.Linear(self.flattened, 512),
            nn.ReLU(),
            nn.Linear(512, 128)
        )

    def forward_once(self, x):
        x = self.backbone(x)
        x = x.view(x.size(0), -1)
        return F.normalize(self.fc(x), p=2, dim=1)

    def forward(self, img1, img2):
        return self.forward_once(img1), self.forward_once(img2)

# ==============================
# Contrastive Loss
# ==============================
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def forward(self, output1, output2, label):
        dists = F.pairwise_distance(output1, output2, p=2)
        loss = label * dists.pow(2) + (1 - label) * F.relu(self.margin - dists).pow(2)
        return loss.mean()


# ==============================
# Train (clean only)
# ==============================
def train(model, loader, criterion, optimizer, epochs=10):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for img1, img2, label in tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}"):
            img1, img2, label = img1.to(device), img2.to(device), label.to(device)
            out1, out2 = model(img1, img2)
            loss = criterion(out1, out2, label)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch [{epoch+1}/{epochs}] Avg Loss: {total_loss / len(loader):.4f}")

# ==============================
# Evaluation: Clean
# ==============================
def evaluate(model, loader, threshold=0.7):
    model.eval()
    y_true, y_pred, sims = [], [], []
    with torch.no_grad():
        for img1, img2, label in loader:
            img1, img2 = img1.to(device), img2.to(device)
            out1, out2 = model(img1, img2)
            sim = F.cosine_similarity(out1, out2).item()
            pred = 1 if sim > threshold else 0
            y_true.append(int(label.item()))
            y_pred.append(pred)
            sims.append(sim)
    acc = accuracy_score(y_true, y_pred)
    try:
        roc_auc = roc_auc_score(y_true, sims)
    except:
        roc_auc = None
    print("\nEvaluation (Clean Data):")
    print(f"Accuracy: {acc:.4f}")
    print(f"ROC AUC: {roc_auc:.4f}" if roc_auc else "ROC AUC: N/A")
    print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))


# ==============================
# Dataset, Model, Train, Evaluate
# ==============================
base_path = './AdvLFW/images' 
dataset = SiameseFaceDataset(base_path, transform=transform)

# Podział na trening/test 80/20
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size], generator=torch.Generator().manual_seed(42))

# Loadery
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

model = SiameseNetwork().to(device)
criterion = ContrastiveLoss(margin=1.0)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Run Training and Evaluation
train(model, train_loader, criterion, optimizer, epochs=10)
evaluate(model, test_loader, threshold=0.819)


/Users/akaszynska/.local/share/virtualenvs/akaszynska-UBELCw3b/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/akaszynska/.local/share/virtualenvs/akaszynska-UBELCw3b/lib/python3.13/site-packages/albumentations/__init__.py:28: UserWarning: A new version of Albumentations is available: '2.0.7' (you have '2.0.6'). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


Using device: mps


Epoch 1/10: 100%|█████████████████████████████| 600/600 [00:57<00:00, 10.45it/s]


Epoch [1/10] Avg Loss: 0.2458


Epoch 2/10: 100%|█████████████████████████████| 600/600 [00:57<00:00, 10.40it/s]


...

Epoch [8/10] Avg Loss: 0.0911


Epoch 9/10: 100%|█████████████████████████████| 600/600 [00:50<00:00, 11.82it/s]


Epoch [9/10] Avg Loss: 0.0665


Epoch 10/10: 100%|████████████████████████████| 600/600 [00:50<00:00, 11.92it/s]


Epoch [10/10] Avg Loss: 0.0454

Evaluation (Clean Data):
Accuracy: 0.7458
ROC AUC: 0.8323
Confusion Matrix:
 [[478 132]
 [173 417]]


In [2]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
import random
import re

# ==============================
# Device Setup
# ==============================
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==============================
# Haar Cascade Face Detection
# ==============================
haar_cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
face_cascade = cv2.CascadeClassifier(haar_cascade_path)

def detect_face_haar(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(40, 40))
    if len(faces) > 0:
        x, y, w, h = faces[0]
        face = image[y:y+h, x:x+w]
        face = cv2.resize(face, (112, 112))
        return face
    else:
        return np.zeros((112, 112, 3), dtype=np.uint8)

# ==============================
# Dataset
# ==============================
class SiameseFaceDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.pairs = []
        self.transform = transform

        for root, _, _ in os.walk(root_dir):
            match = re.search(r'label_(\d+)', root)
            if not match:
                continue
            label = int(match.group(1))
            im0 = next((f for f in os.listdir(root) if f.startswith("im_0_")), None)
            im1 = next((f for f in os.listdir(root) if f.startswith("im_1_")), None)
            if im0 and im1:
                img0_path = os.path.join(root, im0)
                img1_path = os.path.join(root, im1)
                self.pairs.append((img0_path, img1_path, label))

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        img0_path, img1_path, label = self.pairs[idx]
        img0 = cv2.imread(img0_path)
        img1 = cv2.imread(img1_path)
        img0 = cv2.cvtColor(img0, cv2.COLOR_BGR2RGB)
        img1 = cv2.cvtColor(img1, cv2.COLOR_BGR2RGB)
        img0 = detect_face_haar(img0)
        img1 = detect_face_haar(img1)
        if self.transform:
            img0 = self.transform(image=img0)['image']
            img1 = self.transform(image=img1)['image']
        return img0.to(torch.float32), img1.to(torch.float32), torch.tensor(label, dtype=torch.float32)

# ==============================
# Transforms
# ==============================
transform = A.Compose([
    A.Resize(112, 112),
    A.Normalize(),
    ToTensorV2()
])

# ==============================
# Model
# ==============================
class SiameseNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        with torch.no_grad():
            dummy = self.backbone(torch.zeros(1, 3, 112, 112))
            self.flattened = dummy.view(1, -1).shape[1]
        self.fc = nn.Sequential(
            nn.Linear(self.flattened, 512),
            nn.ReLU(),
            nn.Linear(512, 128)
        )

    def forward_once(self, x):
        x = self.backbone(x)
        x = x.view(x.size(0), -1)
        return F.normalize(self.fc(x), p=2, dim=1)

    def forward(self, img1, img2):
        return self.forward_once(img1), self.forward_once(img2)

# ==============================
# Contrastive Loss
# ==============================
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def forward(self, output1, output2, label):
        dists = F.pairwise_distance(output1, output2, p=2)
        loss = label * dists.pow(2) + (1 - label) * F.relu(self.margin - dists).pow(2)
        return loss.mean()


# ==============================
# Train (clean only)
# ==============================
def train(model, loader, criterion, optimizer, epochs=10):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for img1, img2, label in tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}"):
            img1, img2, label = img1.to(device), img2.to(device), label.to(device)
            out1, out2 = model(img1, img2)
            loss = criterion(out1, out2, label)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch [{epoch+1}/{epochs}] Avg Loss: {total_loss / len(loader):.4f}")

# ==============================
# Evaluation: Clean
# ==============================
def evaluate(model, loader, threshold=0.7):
    model.eval()
    y_true, y_pred, sims = [], [], []
    with torch.no_grad():
        for img1, img2, label in loader:
            img1, img2 = img1.to(device), img2.to(device)
            out1, out2 = model(img1, img2)
            sim = F.cosine_similarity(out1, out2).item()
            pred = 1 if sim > threshold else 0
            y_true.append(int(label.item()))
            y_pred.append(pred)
            sims.append(sim)
    acc = accuracy_score(y_true, y_pred)
    try:
        roc_auc = roc_auc_score(y_true, sims)
    except:
        roc_auc = None
    print("\nEvaluation (Clean Data):")
    print(f"Accuracy: {acc:.4f}")
    print(f"ROC AUC: {roc_auc:.4f}" if roc_auc else "ROC AUC: N/A")
    print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))


# ==============================
# Dataset, Model, Train, Evaluate
# ==============================
base_path = './AdvLFW/images' 
dataset = SiameseFaceDataset(base_path, transform=transform)

# Podział na trening/test 80/20
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size], generator=torch.Generator().manual_seed(42))

# Loadery
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

model = SiameseNetwork().to(device)
criterion = ContrastiveLoss(margin=1.0)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Run Training and Evaluation
train(model, train_loader, criterion, optimizer, epochs=10)
evaluate(model, test_loader, threshold=0.7)


Using device: mps


Epoch 1/10: 100%|█████████████████████████████| 600/600 [00:50<00:00, 12.00it/s]


Epoch [1/10] Avg Loss: 0.2444


Epoch 2/10: 100%|█████████████████████████████| 600/600 [00:50<00:00, 11.97it/s]


Epoch [2/10] Avg Loss: 0.2140


...

Epoch [8/10] Avg Loss: 0.1055


Epoch 9/10: 100%|█████████████████████████████| 600/600 [00:50<00:00, 11.90it/s]


Epoch [9/10] Avg Loss: 0.0812


Epoch 10/10: 100%|████████████████████████████| 600/600 [00:50<00:00, 11.85it/s]


Epoch [10/10] Avg Loss: 0.0576

Evaluation (Clean Data):
Accuracy: 0.6575
ROC AUC: 0.8237
Confusion Matrix:
 [[233 377]
 [ 34 556]]


In [3]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
import random
import re

# ==============================
# Device Setup
# ==============================
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==============================
# Haar Cascade Face Detection
# ==============================
haar_cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
face_cascade = cv2.CascadeClassifier(haar_cascade_path)

def detect_face_haar(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(40, 40))
    if len(faces) > 0:
        x, y, w, h = faces[0]
        face = image[y:y+h, x:x+w]
        face = cv2.resize(face, (112, 112))
        return face
    else:
        return np.zeros((112, 112, 3), dtype=np.uint8)

# ==============================
# Dataset
# ==============================
class SiameseFaceDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.pairs = []
        self.transform = transform

        for root, _, _ in os.walk(root_dir):
            match = re.search(r'label_(\d+)', root)
            if not match:
                continue
            label = int(match.group(1))
            im0 = next((f for f in os.listdir(root) if f.startswith("im_0_")), None)
            im1 = next((f for f in os.listdir(root) if f.startswith("im_1_")), None)
            if im0 and im1:
                img0_path = os.path.join(root, im0)
                img1_path = os.path.join(root, im1)
                self.pairs.append((img0_path, img1_path, label))

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        img0_path, img1_path, label = self.pairs[idx]
        img0 = cv2.imread(img0_path)
        img1 = cv2.imread(img1_path)
        img0 = cv2.cvtColor(img0, cv2.COLOR_BGR2RGB)
        img1 = cv2.cvtColor(img1, cv2.COLOR_BGR2RGB)
        img0 = detect_face_haar(img0)
        img1 = detect_face_haar(img1)
        if self.transform:
            img0 = self.transform(image=img0)['image']
            img1 = self.transform(image=img1)['image']
        return img0.to(torch.float32), img1.to(torch.float32), torch.tensor(label, dtype=torch.float32)

# ==============================
# Transforms
# ==============================
transform = A.Compose([
    A.Resize(112, 112),
    A.Normalize(),
    ToTensorV2()
])

# ==============================
# Model
# ==============================
class SiameseNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        with torch.no_grad():
            dummy = self.backbone(torch.zeros(1, 3, 112, 112))
            self.flattened = dummy.view(1, -1).shape[1]
        self.fc = nn.Sequential(
            nn.Linear(self.flattened, 512),
            nn.ReLU(),
            nn.Linear(512, 128)
        )

    def forward_once(self, x):
        x = self.backbone(x)
        x = x.view(x.size(0), -1)
        return F.normalize(self.fc(x), p=2, dim=1)

    def forward(self, img1, img2):
        return self.forward_once(img1), self.forward_once(img2)

# ==============================
# Contrastive Loss
# ==============================
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def forward(self, output1, output2, label):
        dists = F.pairwise_distance(output1, output2, p=2)
        loss = label * dists.pow(2) + (1 - label) * F.relu(self.margin - dists).pow(2)
        return loss.mean()


# ==============================
# Train (clean only)
# ==============================
def train(model, loader, criterion, optimizer, epochs=10):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for img1, img2, label in tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}"):
            img1, img2, label = img1.to(device), img2.to(device), label.to(device)
            out1, out2 = model(img1, img2)
            loss = criterion(out1, out2, label)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch [{epoch+1}/{epochs}] Avg Loss: {total_loss / len(loader):.4f}")

# ==============================
# Evaluation: Clean
# ==============================
from sklearn.metrics import f1_score

def evaluate_with_best_threshold(model, loader):
    model.eval()
    y_true, sims = [], []

    # Collect all true labels and cosine similarities
    with torch.no_grad():
        for img1, img2, label in loader:
            img1, img2 = img1.to(device), img2.to(device)
            out1, out2 = model(img1, img2)
            sim = F.cosine_similarity(out1, out2).item()
            sims.append(sim)
            y_true.append(int(label.item()))

    # Sweep through thresholds to find best
    best_threshold = 0.0
    best_accuracy = 0.0
    best_f1 = 0.0
    for threshold in np.arange(0.0, 1.01, 0.01):
        y_pred = [1 if s > threshold else 0 for s in sims]
        acc = accuracy_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred)
        if acc > best_accuracy:
            best_accuracy = acc
            best_threshold = threshold
            best_f1 = f1

    # Final evaluation with best threshold
    y_pred_final = [1 if s > best_threshold else 0 for s in sims]
    try:
        roc_auc = roc_auc_score(y_true, sims)
    except:
        roc_auc = None

    print("\nEvaluation with Best Threshold:")
    print(f"Best Threshold: {best_threshold:.2f}")
    print(f"Accuracy: {best_accuracy:.4f}")
    print(f"F1 Score: {best_f1:.4f}")
    print(f"ROC AUC: {roc_auc:.4f}" if roc_auc else "ROC AUC: N/A")
    print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred_final))

# ==============================
# Dataset, Model, Train, Evaluate
# ==============================
base_path = './AdvLFW/images' 
dataset = SiameseFaceDataset(base_path, transform=transform)

# Podział na trening/test 80/20
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size], generator=torch.Generator().manual_seed(42))

# Loadery
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

model = SiameseNetwork().to(device)
criterion = ContrastiveLoss(margin=1.0)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Run Training and Evaluation
train(model, train_loader, criterion, optimizer, epochs=10)
evaluate_with_best_threshold(model, test_loader)


Using device: mps


Epoch 1/10: 100%|█████████████████████████████| 600/600 [00:57<00:00, 10.37it/s]


Epoch [1/10] Avg Loss: 0.2387


Epoch 2/10: 100%|█████████████████████████████| 600/600 [00:57<00:00, 10.50it/s]


Epoch [2/10] Avg Loss: 0.2103


...

Epoch [8/10] Avg Loss: 0.0925


Epoch 9/10: 100%|█████████████████████████████| 600/600 [00:58<00:00, 10.25it/s]


Epoch [9/10] Avg Loss: 0.0684


Epoch 10/10: 100%|████████████████████████████| 600/600 [00:58<00:00, 10.33it/s]


Epoch [10/10] Avg Loss: 0.0461

Evaluation with Best Threshold:
Best Threshold: 0.84
Accuracy: 0.7633
F1 Score: 0.7530
ROC AUC: 0.8379
Confusion Matrix:
 [[483 127]
 [157 433]]
